# External Framework Validation — Synthcity (Bank Marketing)

Runs in the isolated `tese_synthcity` conda environment (numpy < 2.0,
opacus==1.4.0). Mirrors the Dataset 1 Synthcity notebook, adapted to the Bank
Marketing feature space. The downstream-performance metric is dropped (single-
class minority reference); the membership-inference/identifiability block keeps
the target.

## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# Folder with the CSV files, relative to this notebook (adjust if needed)
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

from synthcity.plugins.core.dataloader import GenericDataLoader
from synthcity.metrics import Metrics

Data folder: data


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[KeOps] Warning : No C++ compiler found. Define CXX environment variable or install g++.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : No C++ compiler available to check for OpenMP support.
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


## 1. Canonical column definitions (Bank Marketing)

In [2]:
TARGET = "y"

FEATURES = ['age','job','marital','education','default','housing','loan','contact',
            'month','day_of_week','campaign','previous','poutcome','emp.var.rate',
            'cons.price.idx','cons.conf.idx','euribor3m','nr.employed','was_contacted']

# Categorical columns: the 11 nominal categoricals (incl. the binary flag).
# Bank Marketing has NO ordinal columns, so there is a single, unambiguous
# categorical treatment (no Variant A/B, unlike the PAY_X columns of Dataset 1).
CAT_COLS = ['job','marital','education','default','housing','loan','contact',
            'month','day_of_week','poutcome','was_contacted']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 19, f"FEATURES should have 19, got {len(FEATURES)}"
assert TARGET not in FEATURES, "y must not be in FEATURES"
assert len(CAT_COLS) == 11 and len(NUM_COLS) == 8
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))


FEATURES: 19 | CAT: 11 | NUM: 8


## 2. Helpers: feature-only and with-target views

In [3]:
def to_features(df, name):
    d = df.copy()
    if TARGET in d.columns:
        d = d.drop(columns=[TARGET])
    d = d[FEATURES]
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    assert list(d.columns) == FEATURES
    print(f"{name:<16} -> {d.shape}")
    return d


def with_target(df, name):
    """Align to the 19 features AND keep y as the last column.
    The real minority file was saved without y; since it is the minority
    partition, every record is y == 1 by construction, so we add it."""
    d = df.copy()
    if TARGET not in d.columns:
        d[TARGET] = 1                     # minority reference: all y == 1
    cols = FEATURES + [TARGET]
    d = d[cols]
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == cols
    print(f"{name:<16} -> {d.shape}")
    return d

## 3. Load the synthetic datasets and the real minority reference

In [4]:
real_min  = pd.read_csv(DATA_DIR / "bank_real_minority_train.csv")
synth_ctg = pd.read_csv(DATA_DIR / "bank_synthetic_ctgan_final.csv")
synth_ctb = pd.read_csv(DATA_DIR / "bank_synthetic_ctab_final.csv")

print("Feature-only views (no target):")
real_min_p  = to_features(real_min,  "real_minority")
synth_ctg_p = to_features(synth_ctg, "ctgan")
synth_ctb_p = to_features(synth_ctb, "ctabganplus")

print("\nViews with target:")
real_min_t  = with_target(real_min,  "real_minority")
synth_ctg_t = with_target(synth_ctg, "ctgan")
synth_ctb_t = with_target(synth_ctb, "ctabganplus")

for d in (synth_ctg_p, synth_ctb_p):
    assert list(d.columns) == list(real_min_p.columns)
for d in (synth_ctg_t, synth_ctb_t):
    assert list(d.columns) == list(real_min_t.columns)
print("\nAll columns aligned.")

Feature-only views (no target):
real_minority    -> (3248, 19)
ctgan            -> (22335, 19)
ctabganplus      -> (22335, 19)

Views with target:
real_minority    -> (3248, 20)
ctgan            -> (22335, 20)
ctabganplus      -> (22335, 20)

All columns aligned.


## 4. Reconstruct the canonical split → holdout (with target)

In [5]:
df_raw = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=';')

df_c = df_raw.drop(columns=['duration'])
df_c['was_contacted'] = (df_c['pdays'] != 999).astype(int)
df_c = df_c.drop(columns=['pdays'])
df_c[TARGET] = (df_c['y'] == 'yes').astype(int)
df_c['default'] = df_c['default'].replace({'yes': 'unknown'})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "bank_real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
# Compare VALUES, not strict dtypes (int32 vs int64 store the same 0/1 data)
pd.testing.assert_frame_equal(
    train_recon, real_full.reset_index(drop=True),
    check_dtype=False,   # values are compared, not dtypes (int32 vs int64 store the same 0/1 data)
)
print("Split reproduced bit for bit (values). OK.")

# Holdout WITH target, aligned to the with-target schema
holdout_df = X_test.copy()
holdout_df[TARGET] = y_test
holdout_t = with_target(holdout_df, "holdout")
assert list(holdout_t.columns) == FEATURES + [TARGET]
print("Holdout ready:", holdout_t.shape)

Split reproduced bit for bit (values). OK.
holdout          -> (12357, 20)
Holdout ready: (12357, 20)


## 5. Minority-only holdout for the membership-inference reference

In [6]:
# DomiasMIA assumes members and non-members share the SAME distribution.
# Generation reference is minority-only, so the non-member set is too.
holdout_min = holdout_df[holdout_df[TARGET] == 1].copy()
holdout_min_t = with_target(holdout_min, "holdout_minority")
print("Minority holdout:", holdout_min_t.shape)

holdout_minority -> (1392, 20)
Minority holdout: (1392, 20)


## 6. Wrap the data in Synthcity `GenericDataLoader`

In [7]:
def make_loader(df, target=None):
    """Wrap a dataframe in a GenericDataLoader.

    Synthcity 0.2.12 has no argument to declare discrete columns: its encoder
    treats a column as categorical when it is non-numeric or has fewer than
    15 distinct values.
    """
    return GenericDataLoader(
        df.reset_index(drop=True),
        target_column=target,
    )

# feature-only loaders
loader_real_feat = make_loader(real_min_p)
loader_ctg_feat  = make_loader(synth_ctg_p)
loader_ctb_feat  = make_loader(synth_ctb_p)

# with-target loaders
loader_real_tgt = make_loader(real_min_t, target=TARGET)
loader_ctg_tgt  = make_loader(synth_ctg_t, target=TARGET)
loader_ctb_tgt  = make_loader(synth_ctb_t, target=TARGET)
loader_holdout_min = make_loader(holdout_min_t, target=TARGET)

print("Loaders built.")
print("  real (feat):", loader_real_feat.dataframe().shape)
print("  real (tgt) :", loader_real_tgt.dataframe().shape)
print("  holdout min:", loader_holdout_min.dataframe().shape)

Loaders built.
  real (feat): (3248, 19)
  real (tgt) : (3248, 20)
  holdout min: (1392, 20)


## 7. Metric selection

In [8]:
METRICS_FEAT = {
    "stats":     ["ks_test", "prdc", "alpha_precision"],
    "sanity":    ["nearest_syn_neighbor_distance"],
    "detection": ["detection_xgb"],
}

# Privacy only. Downstream performance dropped: minority reference is single-class.
METRICS_TGT = {
    "privacy": ["identifiability_score"],
}

print("Feature-level metrics:", METRICS_FEAT)
print("Target-level metrics :", METRICS_TGT)

Feature-level metrics: {'stats': ['ks_test', 'prdc', 'alpha_precision'], 'sanity': ['nearest_syn_neighbor_distance'], 'detection': ['detection_xgb']}
Target-level metrics : {'privacy': ['identifiability_score']}


## 8. Fidelity / sanity / detection (no target)

In [9]:
def eval_block(loader_real, loader_synth, metrics_dict, model_name):
    print(f"\n{'='*55}\n{model_name}\n{'='*55}")
    res = Metrics.evaluate(
        loader_real, loader_synth, metrics=metrics_dict,
        task_type="classification", random_state=RANDOM_STATE,
        workspace="synthcity_workspace",
    )
    out = res.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "model", model_name)
    return out

fid_ctgan = eval_block(loader_real_feat, loader_ctg_feat, METRICS_FEAT, "CTGAN")
fid_ctab  = eval_block(loader_real_feat, loader_ctb_feat, METRICS_FEAT, "CTAB-GAN+")

fidelity_all = pd.concat([fid_ctgan, fid_ctab], ignore_index=True)
print("\nFidelity / sanity / detection (mean values):")
print(fidelity_all.pivot(index="model", columns="metric", values="mean").round(4))


CTGAN

CTAB-GAN+

Fidelity / sanity / detection (mean values):
metric     detection.detection_xgb.mean  \
model                                     
CTAB-GAN+                           1.0   
CTGAN                               1.0   

metric     sanity.nearest_syn_neighbor_distance.mean  \
model                                                  
CTAB-GAN+                                     0.1339   
CTGAN                                         0.2801   

metric     stats.alpha_precision.authenticity_OC  \
model                                              
CTAB-GAN+                                 0.7081   
CTGAN                                     0.6860   

metric     stats.alpha_precision.authenticity_naive  \
model                                                 
CTAB-GAN+                                    0.7343   
CTGAN                                        0.7491   

metric     stats.alpha_precision.delta_coverage_beta_OC  \
model                                            

## 9. Privacy metrics (with target)

In [10]:
def eval_block_tgt(loader_synth, model_name):
    print(f"\n{'='*55}\n{model_name}\n{'='*55}")
    res = Metrics.evaluate(
        loader_real_tgt, loader_synth,
        X_train=loader_real_tgt, X_ref_syn=loader_holdout_min,
        metrics=METRICS_TGT, task_type="classification",
        random_state=RANDOM_STATE, workspace="synthcity_workspace",
    )
    out = res.reset_index().rename(columns={"index": "metric"})
    out.insert(0, "model", model_name)
    return out

tgt_ctgan = eval_block_tgt(loader_ctg_tgt, "CTGAN")
tgt_ctab  = eval_block_tgt(loader_ctb_tgt, "CTAB-GAN+")

privacy_all = pd.concat([tgt_ctgan, tgt_ctab], ignore_index=True)
print("\nPrivacy (mean values):")
print(privacy_all.pivot(index="model", columns="metric", values="mean").round(4))


CTGAN

CTAB-GAN+

Privacy (mean values):
metric     privacy.identifiability_score.score  \
model                                            
CTAB-GAN+                               0.0902   
CTGAN                                   0.0483   

metric     privacy.identifiability_score.score_OC  
model                                              
CTAB-GAN+                                  0.2152  
CTGAN                                      0.1943  


## 10. Consolidate and save

In [11]:
fidelity_all.to_csv(DATA_DIR / "bank_synthcity_fidelity_results.csv", index=False)
privacy_all.to_csv(DATA_DIR / "bank_synthcity_privacy_results.csv", index=False)
combined = pd.concat([fidelity_all, privacy_all], ignore_index=True)
combined.to_csv(DATA_DIR / "bank_synthcity_all_results.csv", index=False)
print("Saved bank_synthcity_*.csv")

Saved bank_synthcity_*.csv


---

In [12]:
print("=== FIDELITY / SANITY / DETECTION ===")
print(fidelity_all.pivot(index="model", columns="metric", values="mean").round(4))
print("\n=== PRIVACY ===")
print(privacy_all.pivot(index="model", columns="metric", values="mean").round(4))

=== FIDELITY / SANITY / DETECTION ===
metric     detection.detection_xgb.mean  \
model                                     
CTAB-GAN+                           1.0   
CTGAN                               1.0   

metric     sanity.nearest_syn_neighbor_distance.mean  \
model                                                  
CTAB-GAN+                                     0.1339   
CTGAN                                         0.2801   

metric     stats.alpha_precision.authenticity_OC  \
model                                              
CTAB-GAN+                                 0.7081   
CTGAN                                     0.6860   

metric     stats.alpha_precision.authenticity_naive  \
model                                                 
CTAB-GAN+                                    0.7343   
CTGAN                                        0.7491   

metric     stats.alpha_precision.delta_coverage_beta_OC  \
model                                                     
CTAB-GAN+       